In [ ]:
import os
import pandas as pd
from io import StringIO
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np

In [ ]:

# Chemins d'accès
CTD_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Données brutes\CTD"
VUSITU_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Données brutes\TROLL"
BARO_PATH = r'Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx'
OLDDATA_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_consolide_OLD.xlsx"
NEW_CONSOLIDATED_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_consolide_updated.xlsx"
METADATA_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan"

In [ ]:

# Fonction pour lire les fichiers ctd
def lire_fichier_CTD(filenum, path, skiprows=63):
    # Rechercher le fichier correspondant
    nom_fichier = [f for f in os.listdir(path) if f"Cabouy_{filenum}" in f and f"Cabouy_{filenum}" == f[:len(f"Cabouy_{filenum}")]]
    if not nom_fichier:
        raise FileNotFoundError(f"Aucun fichier trouvé pour Cabouy_{filenum} dans {path}")
    
    # Essayer d'abord de lire avec utf-8, puis fallback en latin1 si une erreur survient
    try:
        df = pd.read_csv(os.path.join(path, nom_fichier[0]), sep=';', encoding='utf-8', skiprows=skiprows)
    except UnicodeDecodeError:
        df = pd.read_csv(os.path.join(path, nom_fichier[0]), sep=';', encoding='latin1', skiprows=skiprows)
    
    # Supprimer la dernière ligne si elle est vide ou incorrecte
    df = df.iloc[:-1]
    
    # Conversion explicite des dates
    sample_date = df['Date/time'].iloc[0]
    if '/' in sample_date and sample_date[2] == '/':  
        df['Date/time'] = pd.to_datetime(df['Date/time'], dayfirst=True, errors='coerce')
    else: 
        df['Date/time'] = pd.to_datetime(df['Date/time'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
    
    # Arrondir les dates à l'heure la plus proche
    df['Date/time'] = df['Date/time'].dt.round('h')
    
    # Gestion de la conductivité : conversion en µS/cm si nécessaire
    if '2:Cond. spéc.[ms/cm]' in df.columns:
        # Appliquer la correction uniquement si le type est 'object' (chaîne de caractères)
        if df['2:Cond. spéc.[ms/cm]'].dtype == 'object':
            df['2:Cond. spéc.[ms/cm]'] = df['2:Cond. spéc.[ms/cm]'].str.replace(',', '.').str.strip()
        # Convertir en numérique et multiplier par 1000 pour µS/cm
        df['2:Cond. spéc.[ms/cm]'] = pd.to_numeric(df['2:Cond. spéc.[ms/cm]'], errors='coerce') * 1000
        df.rename(columns={'2:Cond. spéc.[ms/cm]': '2:Cond. spéc.[µS/cm]'}, inplace=True)
    
    # Remplacement des virgules par des points et conversion en numérique pour les autres colonnes
    for col in df.columns[1:]:  # Ignorer la colonne 'Date/time'
        if df[col].dtype == 'object':  # Vérifier si la colonne contient des chaînes de caractères
            df[col] = df[col].str.replace(',', '.').str.strip()  # Remplacer les virgules et enlever les espaces
        df[col] = pd.to_numeric(df[col], errors='coerce')  # Convertir en float en ignorant les erreurs
    
    return df

# Fonction pour lire et nettoyer un fichier VuSitu avec gestion de l'encodage
def lire_fichier_vusitu(filenum, path, skiprows=0):
    # Rechercher le fichier correspondant
    nom_fichier = [f for f in os.listdir(path) if f"VuSitu_{filenum}" in f]
    if not nom_fichier:
        raise FileNotFoundError(f"Aucun fichier trouvé pour VuSitu_{filenum} dans {path}")
    
    # Lecture du fichier avec tentative en utf-8, puis latin1 si échec
    file_path = os.path.join(path, nom_fichier[0])
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
    except UnicodeDecodeError:
        print(f"Échec de la lecture en utf-8 pour {nom_fichier[0]}, tentative avec latin1...")
        with open(file_path, 'r', encoding='latin1') as file:
            lines = file.readlines()
    
    # Enlever les guillemets redondants de chaque ligne
    cleaned_lines = ''.join([line.replace('"', '') for line in lines])

    # Utiliser StringIO pour lire les lignes nettoyées avec pandas
    df = pd.read_csv(StringIO(cleaned_lines), sep=',', skiprows=skiprows)

    # Conversion explicite de la colonne 'Date Heure' en datetime, puis arrondi à l'heure la plus proche
    if 'Date Heure' in df.columns:
        df['Date Heure'] = pd.to_datetime(df['Date Heure'], errors='coerce')
        df['Date Heure'] = df['Date Heure'].dt.round('h')  # Arrondir à l'heure la plus proche

    return df

def renommer_colonnes(df):
    colonnes_a_renommer = {
        # Colonnes existantes à renommer
        'Conductivité spécifique (µS/cm) (737318)': 'Conductivité spécifique (µS/cm)',
        'Conductivité spécifique (µS/cm) (736865)': 'Conductivité spécifique (µS/cm)',
        'Conductivité spécifique (µS/cm) (736870)': 'Conductivité spécifique (µS/cm)',
        'Conductivité spécifique (µS/cm) (736866)': 'Conductivité spécifique (µS/cm)',
        'Turbidité (NTU) (736542)': 'Turbidité (NTU)',
        'Turbidité (NTU) (736534)': 'Turbidité (NTU)',
        'Turbidité (NTU) (736551)': 'Turbidité (NTU)',
        'Turbidité (NTU) (736536)': 'Turbidité (NTU)',
        'Fluorescence de chlorophylle-a (RFU) (740890)': 'Fluorescence de chlorophylle-a (RFU)',
        'Fluorescence de chlorophylle-a (RFU) (740879)': 'Fluorescence de chlorophylle-a (RFU)',
        'Fluorescence de chlorophylle-a (RFU) (740919)': 'Fluorescence de chlorophylle-a (RFU)',
        'Fluorescence de chlorophylle-a (RFU) (740874)': 'Fluorescence de chlorophylle-a (RFU)',
        'Concentration de chlorophylle-a (µg/L) (740890)': 'Concentration de chlorophylle-a (µg/L)',
        'Concentration de chlorophylle-a (µg/L) (740874)': 'Concentration de chlorophylle-a (µg/L)',
        'Concentration de chlorophylle-a (µg/L) (740879)' :'Concentration de chlorophylle-a (µg/L)',
        'Concentration RDO (mg/L) (735988)': 'Concentration RDO (mg/L)',
        'Concentration RDO (mg/L) (735815)': 'Concentration RDO (mg/L)',
        'Concentration RDO (mg/L) (735819)': 'Concentration RDO (mg/L)',
        'Concentration RDO (mg/L) (735995)': 'Concentration RDO (mg/L)',
        'Saturation RDO (%Sat) (735988)': 'Saturation RDO (%Sat)',
        'Saturation RDO (%Sat) (735815)': 'Saturation RDO (%Sat)',
        'Température (°C) (740386)': 'Température (°C)',
        'Température (°C) (740405)': 'Température (°C)',
        'Température (°C) (740282)': 'Température (°C)',
        'Température (°C) (740345)': 'Température (°C)',
    }
    
    df.rename(columns=colonnes_a_renommer, inplace=True)
    
    return df

In [ ]:
def convertir_en_utc0(df, nom_fichier, metadata):
    # Récupérer le fuseau horaire
    if nom_fichier in metadata['Nom fichier'].values:
        utc_fuseau = metadata.loc[metadata['Nom fichier'] == nom_fichier, 'UTC Fichier'].values[0]
    else:
        raise ValueError(f"Fuseau horaire non trouvé pour le fichier : {nom_fichier}")

    # Décalage en heures selon le fuseau horaire
    decalage = 1 if utc_fuseau == 'UTC+1' else 2 if utc_fuseau == 'UTC+2' else 0

    # Ajouter une colonne pour les dates en UTC+0
    df['Date/time (UTC+0)'] = df['Date/time'] - pd.Timedelta(hours=decalage)
    return df
    
# Fichier UTC
metadata_file = os.path.join(METADATA_PATH, "UTC_CTD.xlsx")
metadata = pd.read_excel(metadata_file)

# Lire les données barométriques à partir du fichier Excel
baro_data = pd.read_excel(BARO_PATH)

# Initialiser un DataFrame vide pour stocker les données fusionnées
merge_ctd_df = pd.DataFrame()

# Boucle pour traiter les fichiers CTD
for num in range(1, 49+1):
    filenum = str(num)  # Convertir le numéro en chaîne

    # Lire les données CTD
    CTD = lire_fichier_CTD(filenum, CTD_PATH, skiprows=63)

    # Récupérer le nom du fichier pour correspondre aux métadonnées
    nom_fichier = [f for f in os.listdir(CTD_PATH) if f"Cabouy_{filenum}" in f]
    if not nom_fichier:
        print(f" Aucun fichier trouvé pour Cabouy_{filenum}")
        continue
    nom_fichier = nom_fichier[0]  # Prendre le premier fichier trouvé

    # Appliquer la correction UTC+0
    CTD = convertir_en_utc0(CTD, nom_fichier, metadata)

    # Supprimer l'ancienne colonne 'Date/time' et renommer 'Date/time (UTC+0)' → 'Date/time'
    CTD.drop(columns=['Date/time'], inplace=True)
    CTD.rename(columns={'Date/time (UTC+0)': 'Date/time'}, inplace=True)

    # Fusionner avec les données Baro sur la base des dates (qui sont maintenant en UTC+0)
    merged_df = pd.merge(CTD, baro_data[['DATE', 'Patm Ouysse Calès [hPa]']], left_on='Date/time', right_on='DATE', how='left')

    # Calcul de la hauteur piézométrique : correction par soustraction de la pression barométrique
    merged_df['Niveau_(cm)'] = merged_df['Pression[cmH2O]'] - merged_df['Patm Ouysse Calès [hPa]']
    
    # Renommer la colonne de conductivité et température pour les rendre plus claires
    merged_df.rename(columns={'2:Cond. spéc.[µS/cm]': 'Cond_(µS/cm)', 'Température[°C]': 'Temp_(°C)'}, inplace=True)

    # Conserver uniquement les colonnes utiles
    merged_df = merged_df[['Date/time', 'Niveau_(cm)', 'Pression[cmH2O]', 'Patm Ouysse Calès [hPa]', 'Cond_(µS/cm)', 'Temp_(°C)']]

    # Ajouter les données fusionnées au DataFrame final
    merge_ctd_df = pd.concat([merge_ctd_df, merged_df], ignore_index=True)

merge_ctd_df


In [ ]:
#correction shift des anciennes données avec les nouvelles
# Convertir 'DATE' des deux DataFrames en format datetime
olddata_df = pd.read_excel(OLDDATA_PATH)
olddata_df['DATE'] = pd.to_datetime(olddata_df['DATE'])
merge_ctd_df['DATE'] = pd.to_datetime(merge_ctd_df['Date/time'])

# Step 2: Identifier les dates de jonction
last_date_old = olddata_df['DATE'].max()  # Dernière date dans l'ancien fichier
first_date_new = merge_ctd_df['DATE'].min()  # Première date dans le nouveau fichier

print(f"Dernière date dans les anciennes données : {last_date_old}")
print(f"Première date dans les nouvelles données : {first_date_new}")

# Step 3: Filtrer les données autour de la jonction (7 jours avant la dernière ancienne date et 7 jours après la première nouvelle date)
old_data_filtered = olddata_df[olddata_df['DATE'] >= (last_date_old - pd.Timedelta(days=1))]
new_data_filtered = merge_ctd_df[merge_ctd_df['DATE'] <= (first_date_new + pd.Timedelta(days=1))]

# Step 4: Calculer le décalage pour le niveau d'eau (Niveau)
dernier_niveau_old = old_data_filtered['Niveau_(cm)'].dropna().iloc[-1]  # Dernier niveau d'eau valide dans les anciennes données
premier_niveau_new = new_data_filtered['Niveau_(cm)'].dropna().iloc[0]  # Premier niveau d'eau valide dans les nouvelles données
shift_niveau = dernier_niveau_old - premier_niveau_new  # Calculer le décalage

# Appliquer le décalage aux nouvelles données
merge_ctd_df['Niveau_corr'] = merge_ctd_df['Niveau_(cm)'] + shift_niveau

# Step 5: Calculer le décalage pour la conductivité (Conducti)
derniere_cond_old = old_data_filtered['Cond_CTD_(µS/cm)'].dropna().iloc[-1]  # Dernière conductivité valide dans les anciennes données
premiere_cond_new = new_data_filtered['Cond_(µS/cm)'].dropna().iloc[0]  # Première conductivité valide dans les nouvelles données
shift_cond = derniere_cond_old - premiere_cond_new  # Calculer le décalage pour la conductivité

# Appliquer le décalage aux nouvelles données pour la conductivité
merge_ctd_df['Conducti_corr'] = merge_ctd_df['Cond_(µS/cm)'] + shift_cond

# Graphique pour controler la correction 
# Fenetre de 7 jours autour de la jointure
new_data_filtered_corr = merge_ctd_df[merge_ctd_df['DATE'] <= (first_date_new + pd.Timedelta(days=7))]
fig, ax1 = plt.subplots(figsize=(11, 3))
ax1.plot(old_data_filtered['DATE'], old_data_filtered['Niveau_(cm)'], label='Niveau (Consolidé)', color='green')
ax1.plot(new_data_filtered_corr['DATE'], new_data_filtered_corr['Niveau_corr'], label='Niveau (Nouveau - Corrigé)', color='blue')
ax1.set_ylabel('Niveau (cm)')
ax1.tick_params(axis='y')

ax2 = ax1.twinx()
ax2.plot(old_data_filtered['DATE'], old_data_filtered['Cond_CTD_(µS/cm)'], label='Conductivité (Consolidé)', color='orange')
ax2.plot(new_data_filtered_corr['DATE'], new_data_filtered_corr['Conducti_corr'], label='Conductivité (Nouveau - Corrigé)', color='red')
ax2.set_ylabel('Conductivité (µS/cm)')
ax2.tick_params(axis='y')

fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=4)
plt.tight_layout()
plt.show()


In [ ]:
# Charger les métadonnées
metadata_troll = pd.read_excel(os.path.join(METADATA_PATH, "UTC_Troll.xlsx"), sheet_name=0)

merge_troll_df = pd.DataFrame()

for num in range(1, 50+1):
    try:
        filenum = f"{num:02d}"      
        df_vusitu = lire_fichier_vusitu(filenum, VUSITU_PATH)        
        df_vusitu = renommer_colonnes(df_vusitu)

        # Trouver le fuseau horaire du fichier
        nom_fichier_vusitu = [f for f in os.listdir(VUSITU_PATH) if f"VuSitu_{filenum}" in f]
        if not nom_fichier_vusitu:
            raise FileNotFoundError(f"Aucun fichier VuSitu trouvé pour {filenum}")
        nom_fichier_vusitu = nom_fichier_vusitu[0]

        if nom_fichier_vusitu in metadata_troll['Nom fichier'].values:
            utc_fuseau = metadata_troll.loc[metadata_troll['Nom fichier'] == nom_fichier_vusitu, 'UTC Fichier'].values[0]
            decalage = {'UTC+1': 1, 'UTC+2': 2}.get(utc_fuseau, 0)

            # Appliquer la conversion en UTC+0
            df_vusitu['Date/time (UTC+0)'] = df_vusitu['Date Heure'] - pd.Timedelta(hours=decalage)

            # Remplacer l'ancienne colonne 'DATE' par la nouvelle version UTC+0
            df_vusitu.drop(columns=['Date Heure'], inplace=True, errors='ignore')
            df_vusitu.rename(columns={'Date/time (UTC+0)': 'Date Heure'}, inplace=True)

        merge_troll_df = pd.concat([merge_troll_df, df_vusitu], ignore_index=True)

    except FileNotFoundError as e:
        print(e)

merge_troll_df


In [ ]:
# Valeur de niveau NGF de référence pour Cabouy
niveau_ngf_Cabouy = 107.6158

# Conversion des colonnes 'DATE' en datetime pour l'ensemble des DataFrames
olddata_df['DATE'] = pd.to_datetime(olddata_df['DATE'], errors='coerce')
merge_ctd_df['DATE'] = pd.to_datetime(merge_ctd_df['DATE'], errors='coerce')
merge_troll_df['Date Heure'] = pd.to_datetime(merge_troll_df['Date Heure'], errors='coerce')

# Renommer les colonnes des nouvelles données CTD pour correspondre au format attendu
merge_ctd_df_renamed = merge_ctd_df[['DATE','Niveau_(cm)', 'Cond_(µS/cm)', 'Temp_(°C)']].rename(
    columns={
        'Niveau_(cm)':'Niveau_(cm)',
        'Cond_(µS/cm)': 'Cond_CTD_(µS/cm)', 
        'Temp_(°C)': 'Temp _CTD(°C)',
    })

# Renommer les colonnes des données Aquatroll (VuSitu) pour correspondre au format attendu
merge_troll_df_renamed = merge_troll_df.rename(columns={
    'Date Heure' : 'DATE',
    'Turbidité (NTU)': 'Turbidity_Troll_(NTU)',
    'Concentration RDO (mg/L)': 'O2_Troll_(mg/l)',
    'Fluorescence de chlorophylle-a (RFU)': 'FluorescenceChloro_a_Troll_(RFU)',
    'Concentration de chlorophylle-a (µg/L)': 'ConcentrationChloro_a_(µg/l)',
    'Conductivité spécifique (µS/cm)': 'Cond_Troll_(µS/cm)',
    'Température (°C)': 'température_Troll_(°C)',
    'Saturation RDO (%Sat)': 'O2 (%Sat)'
})

# Fusionner les données CTD et Aquatroll (VuSitu) sur la base des dates
merged_new_data = pd.merge(merge_ctd_df_renamed, merge_troll_df_renamed, on='DATE', how='outer')

# Calcul de 'Niveau_NGF' : à partir de 'Niveau' selon la formule donnée
merged_new_data['Niveau_(mNGF)'] = niveau_ngf_Cabouy - (niveau_ngf_Cabouy - merged_new_data['Niveau_(cm)']) / 100

# Concaténer les anciennes données avec les nouvelles données fusionnées
full_data = pd.concat([olddata_df, merged_new_data], ignore_index=True)

# Supprimer les colonnes dupliquées après la concaténation
full_data = full_data.loc[:, ~full_data.columns.duplicated()]

# Trier les données par date pour garantir un ordre chronologique
full_data = full_data.sort_values(by='DATE').reset_index(drop=True)

# Afficher les données finales
full_data


In [ ]:
# Sauvegarder les nouvelles données consolidées dans un fichier Excel
full_data.to_excel(NEW_CONSOLIDATED_PATH, index=False)
print(f"Nouvelles données consolidées enregistrées sous : {NEW_CONSOLIDATED_PATH}")

In [ ]:
# Chemin vers les fichiers de données consolidées et de mesure ponctuelle
new_consolidated_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_consolide_updated.xlsx")
punctual_data_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\punctual_measurements.xlsx")

# Charger les données consolidées et ponctuelles
new_consolidated_df = pd.read_excel(new_consolidated_path)
punctual_df = pd.read_excel(punctual_data_path)

# Conversion des dates en datetime
new_consolidated_df['DATE'] = pd.to_datetime(new_consolidated_df['DATE'])
punctual_df['Datetime'] = pd.to_datetime(punctual_df['Jour'], dayfirst=True, errors='coerce')

# Supprimer les données dans la période spécifiée
start_date = pd.to_datetime("2019-10-14 17:00")
end_date = pd.to_datetime("2020-02-25 17:00")
new_consolidated_df = new_consolidated_df[~((new_consolidated_df['DATE'] >= start_date) & (new_consolidated_df['DATE'] <= end_date))]

# Filtrer les valeurs de 'Niveau' inférieures à 33 cm
new_consolidated_df = new_consolidated_df[new_consolidated_df['Niveau_(cm)'] >= 33]
# Ajouter 40 cm à toute la chronique
new_consolidated_df['Niveau_(cm)'] = new_consolidated_df['Niveau_(cm)'] + 56.5 # la Nouvelle echelle est 48cm plus haut que l'ancienne

# Apply corrections based on control points
for _, row in punctual_df.iterrows():
    point_date = row['Datetime']
    point_value = row['Hauteur (cm)']
    apply_correction = row.get("Correction", "Non")  # Assume "Non" if the column is missing

    if pd.isna(point_value) or apply_correction != "Oui":
        continue  # Skip points without correction or invalid values

    # Find the nearest measurement in consolidated data
    nearest_measure_index = (new_consolidated_df['DATE'] - point_date).abs().idxmin()
    nearest_measure_value = new_consolidated_df.at[nearest_measure_index, 'Niveau_(cm)']

    # Check proximity (within 1 hour)
    if pd.isna(nearest_measure_value) or abs(
        (new_consolidated_df['DATE'][nearest_measure_index] - point_date).total_seconds()
    ) > 3600:
        continue

    # Calculate and apply the offset
    decalage = point_value - nearest_measure_value
    print(f"Date : {point_date}, Décalage appliqué : {decalage} cm")

    # Apply the offset to all subsequent measurements
    new_consolidated_df.loc[new_consolidated_df['DATE'] >= point_date, 'Niveau_(cm)'] += decalage
    
# Créer un graphique interactif avec Plotly
fig = go.Figure()

# Ajouter la ligne pour le niveau d'eau ('Niveau_(cm)') dans les données consolidées
fig.add_trace(go.Scatter(
    x=new_consolidated_df['DATE'], 
    y=new_consolidated_df['Niveau_(cm)'], 
    mode='lines', 
    name='Niveau_(cm)',
    line=dict(color='grey', width=1.5),
    opacity=0.5
))

# Ajouter les points de contrôle ponctuels (Hauteur en cm) avec des marqueurs
fig.add_trace(go.Scatter(
    x=punctual_df['Datetime'], 
    y=punctual_df['Hauteur (cm)'],
    mode='markers', 
    name='Points de contrôle',
    marker=dict(color='blue', symbol='x', size=8)
))

# Configurer les axes et les étiquettes
fig.update_layout(
    title='Visualisation du niveau d\'eau avec points de contrôle après suppression de la période',
    xaxis_title='Date',
    yaxis_title='Niveau (cm)',
    legend_title='Type de données',
    template='plotly_white'
)
# Afficher le graphique
fig.show()

# Sauvegarder les données corrigées dans un fichier Excel
output_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Niveau_Period_Supprimee.xlsx")
new_consolidated_df.to_excel(output_path, index=False)
print(f"Données nettoyées sauvegardées dans : {output_path}")


In [ ]:
# Chemin vers les fichiers de données
data_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Niveau_Period_Supprimee.xlsx"
)
punctual_data_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\punctual_measurements.xlsx"
)

# Charger les données
data = pd.read_excel(data_path)
punctual_df = pd.read_excel(punctual_data_path)

# Conversion des dates en datetime
data['DATE'] = pd.to_datetime(data['DATE'], errors='coerce')
punctual_df['Datetime'] = pd.to_datetime(punctual_df['Jour'], dayfirst=True, errors='coerce')

# Supprimer les valeurs inférieures à 10 et les valeurs négatives dans les colonnes 'Cond_Troll_(µS/cm)' et 'Cond_CTD_(µS/cm)'
data.loc[data['Cond_Troll_(µS/cm)'] < 30, 'Cond_Troll_(µS/cm)'] = None
data.loc[data['Cond_Troll_(µS/cm)'] < 0, 'Cond_Troll_(µS/cm)'] = None
data.loc[data['Cond_CTD_(µS/cm)'] < 30, 'Cond_CTD_(µS/cm)'] = None
data.loc[data['Cond_CTD_(µS/cm)'] < 0, 'Cond_CTD_(µS/cm)'] = None

# Créer la colonne "Conductivité" avec comblement des données manquantes
data['Conductivité'] = None
decalage = 0

# Logique de comblement : Priorité à 'Cond_Troll_(µS/cm)', sinon 'Cond_CTD_(µS/cm)'
for i in range(len(data)):
    if not pd.isna(data.at[i, 'Cond_Troll_(µS/cm)']):  # Utiliser Cond_Troll_(µS/cm) en priorité
        data.at[i, 'Conductivité'] = data.at[i, 'Cond_Troll_(µS/cm)']
        decalage = data.at[i, 'Cond_Troll_(µS/cm)'] - data.at[i, 'Cond_CTD_(µS/cm)']  # Calculer le décalage
    elif not pd.isna(data.at[i, 'Cond_CTD_(µS/cm)']):  # Utiliser Cond_CTD_(µS/cm) si Troll est manquant
        data.at[i, 'Conductivité'] = data.at[i, 'Cond_CTD_(µS/cm)'] + decalage

# Supprimer toute valeur négative restante dans 'Conductivité'
data.loc[data['Conductivité'] < 0, 'Conductivité'] = None

# Sauvegarder les données corrigées
output_corrected_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Niveau_Corrigé.xlsx"
)
data.to_excel(output_corrected_path, index=False)
print(f"Données corrigées sauvegardées dans : {output_corrected_path}")

# Visualisation interactive pour choisir les points de contrôle
fig = go.Figure()

# Ajouter les lignes pour Cond_Troll_(µS/cm), Cond_CTD_(µS/cm) et Conductivité
fig.add_trace(go.Scatter(
    x=data['DATE'], y=data['Cond_Troll_(µS/cm)'], 
    mode='lines', 
    name='Conductivité Troll (prioritaire)', 
    line=dict(color='blue'),
    hovertemplate='Date: %{x}<br>Cond Troll: %{y:.2f} µS/cm<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=data['DATE'], y=data['Cond_CTD_(µS/cm)'], 
    mode='lines', 
    name='Conductivité CTD (secondaire)', 
    line=dict(color='orange'),
    hovertemplate='Date: %{x}<br>Cond CTD: %{y:.2f} µS/cm<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=data['DATE'], y=data['Conductivité'], 
    mode='lines', 
    name='Conductivité corrigée', 
    line=dict(color='green'),
    hovertemplate='Date: %{x}<br>Conductivité corrigée: %{y:.2f} µS/cm<extra></extra>'
))

# Ajouter les points de contrôle
fig.add_trace(go.Scatter(
    x=punctual_df['Datetime'],
    y=punctual_df['Conductivité'],
    mode='markers',
    name='Points de contrôle',
    marker=dict(symbol='x', color='red', size=8),
    hovertemplate='Date: %{x}<br>Point de contrôle: %{y:.2f} µS/cm<extra></extra>'
))

# Configuration du graphique
fig.update_layout(
    title='Visualisation des données de conductivité avec points de contrôle',
    xaxis_title='Date',
    yaxis_title='Conductivité (µS/cm)',
    legend_title='Source de données',
    template='plotly_white',
    hovermode='x unified'
)

# Afficher le graphique
fig.show()

In [ ]:
# Chemin vers les fichiers de données
data_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Niveau_Corrigé.xlsx"
)
punctual_data_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\punctual_measurements_conducti.xlsx"
)

# Charger les données
data = pd.read_excel(data_path)
punctual_df = pd.read_excel(punctual_data_path)

# Conversion des dates en datetime
data['DATE'] = pd.to_datetime(data['DATE'], errors='coerce')
punctual_df['Datetime'] = pd.to_datetime(punctual_df['Jour'], dayfirst=True, errors='coerce')

# Supprimer les valeurs inférieures à 10 et les valeurs négatives dans les colonnes 'Cond_Troll_(µS/cm)' et 'Cond_CTD_(µS/cm)'
data.loc[data['Cond_Troll_(µS/cm)'] < 10, 'Cond_Troll_(µS/cm)'] = None
data.loc[data['Cond_Troll_(µS/cm)'] < 0, 'Cond_Troll_(µS/cm)'] = None
data.loc[data['Cond_CTD_(µS/cm)'] < 10, 'Cond_CTD_(µS/cm)'] = None
data.loc[data['Cond_CTD_(µS/cm)'] < 0, 'Cond_CTD_(µS/cm)'] = None

# Créer la colonne "Conductivité" avec comblement des données manquantes
data['Conductivité'] = None
decalage = 0

# Logique de comblement : Priorité à 'Cond_Troll_(µS/cm)', sinon 'Cond_CTD_(µS/cm)'
for i in range(len(data)):
    if not pd.isna(data.at[i, 'Cond_Troll_(µS/cm)']):  # Utiliser Cond_Troll_(µS/cm) en priorité
        data.at[i, 'Conductivité'] = data.at[i, 'Cond_Troll_(µS/cm)']
        decalage = data.at[i, 'Cond_Troll_(µS/cm)'] - data.at[i, 'Cond_CTD_(µS/cm)']  # Calculer le décalage
    elif not pd.isna(data.at[i, 'Cond_CTD_(µS/cm)']):  # Utiliser Cond_CTD_(µS/cm) si Troll est manquant
        data.at[i, 'Conductivité'] = data.at[i, 'Cond_CTD_(µS/cm)'] + decalage

# Supprimer toute valeur négative restante dans 'Conductivité'
data.loc[data['Conductivité'] < 0, 'Conductivité'] = None

# Appliquer les corrections en fonction des points marqués "Oui" dans la colonne "Correction"
point_colors = []  # Liste pour stocker les couleurs des points de contrôle
for index, row in punctual_df.iterrows():
    point_date = row['Datetime']
    point_value = row['Conductivité']
    apply_correction = row.get("Correction")  # Vérifier la colonne "Correction"

    if pd.isna(point_value) or apply_correction != "Oui":
        point_colors.append('blue')  # Point non utilisé pour la correction
        continue

    # Trouver la mesure la plus proche dans les données consolidées
    nearest_measure_index = (data['DATE'] - point_date).abs().idxmin()
    nearest_measure_value = data.at[nearest_measure_index, 'Conductivité']

    # Si la mesure est trop éloignée dans le temps, ne pas appliquer la correction
    if abs((data['DATE'][nearest_measure_index] - point_date).total_seconds()) > 3600:
        point_colors.append('blue')  # Point non utilisé pour la correction
        continue

    # Calculer le décalage et l'appliquer aux données après ce point de contrôle
    decalage = point_value - nearest_measure_value
    print(f"Date : {point_date}, Décalage appliqué : {decalage} µS/cm")
    data.loc[data['DATE'] >= point_date, 'Conductivité'] += decalage
    point_colors.append('red')  # Point utilisé pour la correction

# Sauvegarder les données corrigées
output_corrected_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_consolidé_et_corrigé.xlsx"
)
data.to_excel(output_corrected_path, index=False)
print(f"Données corrigées sauvegardées dans : {output_corrected_path}")

# Visualisation interactive avec Plotly
fig = go.Figure()

# Ajouter les lignes pour Conductivité corrigée
fig.add_trace(go.Scatter(
    x=data['DATE'], y=data['Conductivité'], 
    mode='lines', 
    name='Conductivité corrigée', 
    line=dict(color='green'),
    hovertemplate='Date: %{x}<br>Conductivité corrigée: %{y:.2f} µS/cm<extra></extra>'
))

# Ajouter les points de contrôle
fig.add_trace(go.Scatter(
    x=punctual_df['Datetime'],
    y=punctual_df['Conductivité'],
    mode='markers',
    name='Points de contrôle',
    marker=dict(symbol='x', color=point_colors, size=8),
    hovertemplate='Date: %{x}<br>Point de contrôle: %{y:.2f} µS/cm<extra></extra>'
))

# Configuration du graphique
fig.update_layout(
    title='Visualisation des données corrigées avec points de contrôle',
    xaxis_title='Date',
    yaxis_title='Conductivité (µS/cm)',
    legend_title='Source de données',
    template='plotly_white',
    hovermode='x unified'
)

# Afficher le graphique
fig.show()

In [ ]:
# Chemin vers le fichier des données
data_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_consolidé_et_corrigé.xlsx"
)

# Charger les données
data = pd.read_excel(data_path)
data['DATE'] = pd.to_datetime(data['DATE'], errors='coerce')

periods_to_remove = [
    ("2023-09-05 13:00", "2023-10-21 22:00","Temp _CTD(°C)"),
    ("2021-04-09 14:00", "2021-06-03 12:00","O2_Troll_(mg/l)")
]

# Convertir 'DATE' en format datetime pour manipulation
data['DATE'] = pd.to_datetime(data['DATE'], errors='coerce')

# Appliquer les suppressions en fonction des périodes définies
for period in periods_to_remove:
    start, end, column = pd.to_datetime(period[0], errors='coerce'), pd.to_datetime(period[1], errors='coerce'), period[2]
    if start > end:  # Corriger les inversions de dates si nécessaire
        start, end = end, start
    data.loc[(data['DATE'] >= start) & (data['DATE'] <= end), column] = None

# Ajouter une colonne pour la version nettoyée
data['Conductivité_cleaned'] = data['Conductivité']

# Appliquer le filtre IQR uniquement jusqu'au 14 février 2021
end_iqr_date = pd.Timestamp('2021-02-14 23:59:59')
data_iqr = data[data['DATE'] <= end_iqr_date]

# Filtre IQR sur les données jusqu'à la date spécifiée
window_size = '800h'
for i in range(len(data_iqr)):
    start_date = data_iqr['DATE'].iloc[i] - pd.Timedelta(window_size)
    end_date = data_iqr['DATE'].iloc[i] + pd.Timedelta(window_size)
    
    # Extraire les données dans cette fenêtre
    window_data = data_iqr[(data_iqr['DATE'] >= start_date) & (data_iqr['DATE'] <= end_date)]['Conductivité']
    
    # Calculer le IQR
    Q1 = window_data.quantile(0.25)
    Q3 = window_data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filtrer les outliers
    if data_iqr['Conductivité'].iloc[i] < lower_bound or data_iqr['Conductivité'].iloc[i] > upper_bound:
        data.at[data_iqr.index[i], 'Conductivité_cleaned'] = None  # Marquer comme NaN

# Interpolation temporelle sur toute la série
data = data.set_index('DATE')
data['Conductivité_cleaned'] = data['Conductivité_cleaned'].interpolate(method='time')

# Calcul de la moyenne mobile glissante sur 6 heures
data['Conductivité_Moyenne_Mobile'] = data['Conductivité_cleaned'].rolling(window='6h', center=True).mean()

# Réinitialiser l'index
data = data.reset_index()

# Créer le graphique interactif
fig = go.Figure()

# Ajouter les données d'origine
fig.add_trace(go.Scatter(
    x=data['DATE'], y=data['Conductivité'],
    mode='lines', name='Conductivité (originale)',
    line=dict(color='orange')
))

# Ajouter les données nettoyées
fig.add_trace(go.Scatter(
    x=data['DATE'], y=data['Conductivité_cleaned'],
    mode='lines', name='Conductivité (nettoyée et interpolée)',
    line=dict(color='green')
))

# Ajouter la moyenne mobile
fig.add_trace(go.Scatter(
    x=data['DATE'], y=data['Conductivité_Moyenne_Mobile'],
    mode='lines', name='Moyenne Mobile (6 heures)',
    line=dict(color='blue')
))

# Configurer le graphique
fig.update_layout(
    title='Visualisation des données de conductivité',
    xaxis_title='Date',
    yaxis_title='Conductivité (µS/cm)',
    legend_title='Type de données'
)

# Afficher le graphique
fig.show()

# Exporter les données nettoyées dans un fichier Excel
output_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Calé_et_Nettoyé.xlsx"
)
data.to_excel(output_path, index=False)
print(f"Données nettoyées sauvegardées dans : {output_path}")

In [ ]:
# Chemins des fichiers de données
rainfall_data_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Saint Sauveur\Gaetan\Données brutes\Pluie_BV_Ouysse.csv"
hydro_data_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Calé_et_Nettoyé.xlsx"

# Chargement des données
rainfall_data = pd.read_csv(rainfall_data_path)
hydro_data = pd.read_excel(hydro_data_path)

# Conversion des colonnes de dates en datetime
rainfall_data['Date'] = pd.to_datetime(rainfall_data['Date'], errors='coerce')
hydro_data['DATE'] = pd.to_datetime(hydro_data['DATE'], errors='coerce')

# Moyenne mobile pour les paramètres
hydro_data['Temp_Moving_Avg'] = hydro_data['Temp _CTD(°C)'].rolling(window=12, center=True).mean()
hydro_data['Niveau_Moving_Avg'] = hydro_data['Niveau_(cm)'].rolling(window=12, center=True).mean()
hydro_data['O2_Moving_Avg'] = hydro_data['O2_Troll_(mg/l)'].rolling(window=24, center=True).mean()
hydro_data['FluoChloro_a_Moving_Avg'] = hydro_data['FluorescenceChloro_a_Troll_(RFU)'].rolling(window=24, center=True).mean()
hydro_data['Turbidity_Moving_Avg'] = hydro_data['Turbidity_Troll_(NTU)'].rolling(window=24, center=True).mean()

# Ajustement des dates pour correspondre à la plage du niveau d'eau avec marge de 15 jours
start_date = hydro_data['DATE'].min() - pd.Timedelta(days=15)
end_date = hydro_data['DATE'].max() + pd.Timedelta(days=15)
rainfall_data = rainfall_data[(rainfall_data['Date'] >= start_date) & (rainfall_data['Date'] <= end_date)]
hydro_data = hydro_data[(hydro_data['DATE'] >= start_date) & (hydro_data['DATE'] <= end_date)]

# Création des graphiques
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw={'hspace': 0.05})

# --- Graphe 1 : Niveau d'eau et Précipitations ---
ax1 = axes[0]
ax2 = ax1.twinx()
ax2.bar(
    rainfall_data['Date'], rainfall_data['Precipitation (mm)'], 
    width=0.8, color='royalblue', align='center'
)
ax2.invert_yaxis()
ax2.set_ylabel('Précipitations (mm)', color='royalblue')
ax2.tick_params(axis='y', labelcolor='royalblue')

ax1.plot(hydro_data['DATE'], hydro_data['Niveau_Moving_Avg'], color='lightseagreen')
ax1.set_ylabel('Niveau (cm)', color='lightseagreen')
ax1.tick_params(axis='y', labelcolor='lightseagreen')
ax1.set_xlim([start_date, end_date]) 

# --- Graphe 2 : Conductivité et Température ---
ax3 = axes[1]
ax4 = ax3.twinx()
ax3.plot(hydro_data['DATE'], hydro_data['Conductivité_Moyenne_Mobile'], color='black')
ax3.set_ylabel('Conductivité (µS/cm)', color='black')
ax3.tick_params(axis='y', labelcolor='black')

ax4.plot(hydro_data['DATE'], hydro_data['Temp_Moving_Avg'], color='crimson')
ax4.set_ylabel('Température (°C)', color='crimson')
ax4.tick_params(axis='y', labelcolor='crimson')
ax3.set_xlim([start_date, end_date])  # Définir la plage x

# --- Graphe 3 : Turbidité, Oxygène et Chlorophylle (Moyenne Mobile) ---
ax5 = axes[2]
ax6 = ax5.twinx()
ax5.plot(hydro_data['DATE'], hydro_data['Turbidity_Moving_Avg'], color='darkorange')
ax5.set_ylabel('Turbidité (NTU)', color='darkorange')
ax5.tick_params(axis='y', labelcolor='darkorange')
ax5.set_ylim(0, 100)


# Ajouter un troisième axe pour la chlorophylle
ax7 = ax5.twinx()
ax7.spines['right'].set_position(('outward', 40))  # Décaler le troisième axe
ax7.plot(hydro_data['DATE'], hydro_data['FluoChloro_a_Moving_Avg'], color='green')
ax7.set_ylabel('Chlorophylle (RFU)', color='green')
ax7.tick_params(axis='y', labelcolor='green')

ax6.plot(hydro_data['DATE'], hydro_data['O2_Moving_Avg'], color='darkmagenta')
ax6.set_ylabel('Oxygène (mg/L)', color='darkmagenta')
ax6.tick_params(axis='y', labelcolor='darkmagenta')
ax5.set_xlim([start_date, end_date])
ax6.set_ylim(5, 20)

# Ajustement des marges et sauvegarde en SVG
plt.subplots_adjust(hspace=0.15)  # Réduire l'espace entre les graphes
svg_output_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Graphes.svg"
plt.savefig(svg_output_path, format='svg', dpi=300)
print(f"Graphique sauvegardé en SVG à : {svg_output_path}")

plt.show()

In [ ]:
# Charger les données
data_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Calé_et_Nettoyé.xlsx"
data = pd.read_excel(data_path)

# Conversion de la colonne 'DATE' en datetime
data['DATE'] = pd.to_datetime(data['DATE'], errors='coerce')

# Supprimer les doublons sur DATE
data = data.drop_duplicates(subset='DATE')

# Réindexer avec toutes les heures manquantes
date_range = pd.date_range(start=data['DATE'].min(), end=data['DATE'].max(), freq='h')
data = data.set_index('DATE').reindex(date_range).reset_index()
data.rename(columns={'index': 'DATE'}, inplace=True)

# Récupérer toutes les colonnes sauf "DATE"
colonnes_a_interpoler = data.select_dtypes(include=[np.number]).columns

# Dictionnaire pour stocker les statuts par colonne
statuts = pd.DataFrame({'DATE': data['DATE']})

for col in colonnes_a_interpoler:
    print(f"📌 Traitement de la colonne : {col}")

    # Identifier les valeurs manquantes pour cette colonne
    data[f'Manquante_{col}'] = data[col].isnull().astype(int)

    # Détecter les groupes de valeurs manquantes
    data[f'Groupe_{col}'] = (data[f'Manquante_{col}'] != data[f'Manquante_{col}'].shift()).cumsum()
    groupe_sizes = data.groupby(f'Groupe_{col}')[f'Manquante_{col}'].sum()
    trous_longs = groupe_sizes[groupe_sizes > 12].index  # Groupes avec >12h de trous

    # Appliquer l'interpolation pour cette colonne
    data[col] = data[col].interpolate(method='linear', limit_direction='both')

    # Supprimer les interpolations pour les trous longs
    data.loc[data[f'Groupe_{col}'].isin(trous_longs), col] = np.nan

    # Ajouter le statut spécifique à cette colonne
    statut_col = f'Statut_{col}'
    statuts[statut_col] = 'Brute'
    statuts.loc[data[col].isnull(), statut_col] = 'Manquante'
    statuts.loc[(data[col].notnull()) & (data[f'Manquante_{col}'] == 1), statut_col] = 'Interpolée'

# Nettoyer les colonnes auxiliaires
cols_to_remove = [col for col in data.columns if col.startswith(('Manquante_', 'Groupe_'))]
data = data.drop(columns=cols_to_remove)

# Fusionner les statuts avec les données finales
data = data.merge(statuts, on="DATE", how="left")

# Sauvegarde
output_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Interpolated.xlsx"
data.to_excel(output_path, index=False)

print(f" Fichier sauvegardé : {output_path}")


In [ ]:
import dash
from dash import dcc, html
import plotly.express as px
import pandas as pd
import plotly.io as pio

# 📥 Charger les données
data_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan\Cabouy_Interpolated.xlsx"
data = pd.read_excel(data_path)

# 🛠 Nettoyage des noms de colonnes (suppression des espaces)
data.columns = [col.strip().replace(" ", "_") for col in data.columns]

# 📌 Exclure DATE et Conductivité_Moyenne_Mobile
excluded_columns = ["DATE", "Conductivité_Moyenne_Mobile"]
parametres = [col for col in data.columns if col not in excluded_columns and not col.startswith("Statut_")]

# 🚀 Créer l'application Dash
app = dash.Dash(__name__)

# 🎨 Mise en page de l'application
app.layout = html.Div([
    html.H1("Visualisation des Données Hydrologiques de la station de Cabouy "),
    dcc.Dropdown(
        id="param-dropdown",
        options=[{"label": param, "value": param} for param in parametres],
        value="Niveau_(cm)"  # Valeur par défaut
    ),
    dcc.Graph(id="graph"),
    html.Div(id="error-message", style={"color": "red", "font-weight": "bold"})
])

# 🔄 Callback pour mettre à jour le graphique
@app.callback(
    [dash.dependencies.Output("graph", "figure"),
     dash.dependencies.Output("error-message", "children")],
    [dash.dependencies.Input("param-dropdown", "value")]
)
def update_graph(param):
    if param not in data.columns:
        return {"data": [], "layout": {"title": f"❌ Erreur : {param} non trouvé"}}, f"⚠️ Erreur : {param} non trouvé."

    # Ajouter une colonne de statut (si elle existe dans les données)
    statut_col = "Statut_" + param if "Statut_" + param in data.columns else None

    # ✅ Convertir en numérique pour éviter erreurs
    data[param] = pd.to_numeric(data[param], errors='coerce')

    # 📊 Création du graphique interactif
    fig = px.scatter(
        data,
        x="DATE",
        y=param,
        color=statut_col,  # On utilise la colonne statut pour colorier les points
        title=f"{param}",
        labels={param: param, "DATE": "Date"},
    )

    # 🔧 Ajustement dynamique de l'axe Y
    fig.update_layout(yaxis=dict(range=[data[param].min(), data[param].max()]))

    # Retourner le graphique et le message d'erreur (si aucune erreur)
    return fig, ""

# ▶️ Lancer l'application
if __name__ == "__main__":
    app.run_server(debug=True)
